# Terminal × product roles (demand / supply centers)

Classify each **Installation × Product** (and optional **complex × Product**) from
`trades_imp_exp_master` over **full history**. Algeciras FO was the discovery
example; the pipeline covers **all products** the same way.

Use `AGGREGATE_BY = "complex"` (default) to score desk complexes from
`data/manual/terminal_desk_map.csv` (Algeciras / Bilbao / Barcelona, …), with
unmapped sites as singletons. Flip to `"installation"` to validate the Kpler map.

**Why a separate notebook from `03`:** role math + a monthly cube are heavier.
Keep `03` for light terminal profiles; use this for product-level typology.

**Defaults**
- Status: `Delivered`
- Grain: Product (not Family)
- Window: full history
- Desk map: `data/manual/terminal_desk_map.csv` (owners, facility types, complexes)

**Units**
- `kb` = `Volume (bbl) / 1000` (absolute; import and export both positive)
- `kbd` = `kb / days_in_month` on the month grain (same idea as the ES/FR/PT tracker)
- Lifetime tables expose average monthly `kbd` by direction

**§7 Owner OD / Sankey:** origin→destination waterborne flows from `kpler_trades.duckdb`, tagged with `terminal_desk_map.csv` and joined to roles from this notebook (surplus→deficit / captive same-owner).

**PyGWalker tip:** your FO chart used monthly X on the event master. Here we walk a
**monthly aggregate** of that master (same chart grain, ~450k rows vs ~1.5M events)
with `kernel_computation=True`. Signed measures (`kb_signed`, `kbd_signed`) keep
imports below zero like `kb_imp_exp` on the raw file.

PyGWalker cells at the end of this notebook are commented — use `03_pygwalker_terminal_demand.ipynb` for interactive exploration.


In [13]:
from __future__ import annotations

import calendar
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import duckdb
import pygwalker as pyg
import sys

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from terminal_desk import DESK_MAP_PATH, complex_map_view, load_desk_map, owners_view

TABLEAU_DIR = ROOT / "data" / "derived" / "tableau"
DERIVED_DIR = ROOT / "data" / "derived"
PARQUET_PATH = TABLEAU_DIR / "trades_imp_exp_master.parquet"
CSV_PATH = TABLEAU_DIR / "trades_imp_exp_master.csv"
DB_PATH = ROOT / "data" / "kpler_trades.duckdb"

OUT_MONTHLY = DERIVED_DIR / "terminal_product_monthly.parquet"
OUT_ROLES = DERIVED_DIR / "terminal_product_roles.parquet"
OUT_ROLES_CSV = DERIVED_DIR / "terminal_product_roles.csv"
OUT_COMPLEX_ROLES = DERIVED_DIR / "complex_product_roles.parquet"
OUT_COMPLEX_ROLES_CSV = DERIVED_DIR / "complex_product_roles.csv"
OUT_PATTERNS = DERIVED_DIR / "terminal_product_patterns.parquet"
OUT_PATTERNS_CSV = DERIVED_DIR / "terminal_product_patterns.csv"
OUT_ENTITY_MAP = DERIVED_DIR / "terminal_entity_map.csv"
OUT_OD_FLOWS = DERIVED_DIR / "owner_od_flows.parquet"
OUT_OD_FLOWS_CSV = DERIVED_DIR / "owner_od_flows.csv"

STATUS = "Delivered"

# Role thresholds (same spirit as the Algeciras visual discovery)
IMPORT_HEAVY_MIN = 0.70
EXPORT_HEAVY_MAX = 0.30
MIN_TOTAL_KB = 500.0  # lifetime floor so thin flows are not labeled as centers

# Working grain for briefing / patterns / walkers:
#   "complex"      -> desk map + singleton (1 installation) entities
#   "installation" -> raw Kpler installation names
AGGREGATE_BY = "complex"
assert AGGREGATE_BY in {"complex", "installation"}

# Briefing charts / tables (FO was only the discovery example — cover all products)
FOCUS_COUNTRY = "Germany"
TOP_N_PER_PRODUCT = 5  # ranked demand/supply centers shown per product

# Desk complexes to highlight in impact / example panels
EXAMPLE_COMPLEXES = [
    "Algeciras Storage",
    "Bilbao Ref+Port",
    "Barcelona Port",
]
# Optional installation drill-down (used when AGGREGATE_BY == "installation")
EXAMPLE_INSTALLATIONS = [
    "Vopak Terquimsa Barcelona",
    "Barcelona 32A",
    "APM Terminals Barcelona",  # kept out of Barcelona Port — El Prat Jet
    "Repsol Petronor Refinery",
    "Bilbao Oil Facilities",
    "Vopak Algeciras",
    "CLH Algeciras Storage",
]

# Pattern monitor: last-3m softness + seasonality context
RECENT_MONTHS = 3
MIN_PATTERN_MONTHS = 24  # need enough history for YoY / z-scores
SOFT_Z = -1.0  # recent 3m vs all prior months
SOFT_SEASONAL_RATIO = 0.70  # recent 3m vs same calendar months in prior years
MIN_SOFT_MEAN_KBD = 5.0  # ignore tiny series when raising soft flags
ROLLING_ROLE_MONTHS = 12

ROLE_COLORS = {
    "demand_center": "#c45c26",
    "supply_center": "#2f6f8f",
    "balanced_hub": "#6b7280",
    "thin": "#d1d5db",
}

# §7 Owner OD / Sankey (waterborne installation→installation)
OD_ORIGIN_COUNTRY = FOCUS_COUNTRY  # focus country: outbound=origin filter, inbound=destination filter
OD_DOMESTIC_ONLY = True  # require destination country == origin country
OD_FOCUS_OWNER = None#"Repsol"  # None = all mapped owners; else filter either end
OD_PRODUCTS = ["Naphtha", "Gasoil", "Gasoline", "Jet", "FO"]  # None = all products
OD_MIN_EDGE_KB = 500.0  # drop thin OD edges from charts/tables
OD_SANKEY_TOP_N = 25  # max links in Sankey (keeps chart readable)

OD_HUB_ENTITY = "Repsol Coruna Refinery"  # installation, complex_id, or complex_name; None = skip
OD_HUB_GRAIN = "installation"  # resolve hub as "installation" or "complex"
OD_HUB_COUNTERPARTY_GRAIN = "complex"  # roll partners up to desk complex when mapped
OD_HUB_TOP_N = 12  # max counterparty×product links per direction in hub charts
PRODUCT_COLORS = {
    "Naphtha": "#8b5cf6",
    "Gasoil": "#2f6f8f",
    "Gasoline": "#c45c26",
    "Jet": "#059669",
    "FO": "#6b7280",
}
OD_ISLAND_ZONES = {
    "Las Palmas",
    "Tenerife",
    "Palma",
    "Arrecife",
    "Arinaga",
    "Salinetas Port",
    "Puerto de la Estaca",
    "Santa Cruz de La Palma",
    "Puerto del Rosario",
    "San Sebastián de La Gomera",
}
FLOW_CLASS_COLORS = {
    "captive_same_owner": "#2f6f8f",
    "internal_balancing": "#1f4e66",
    "seller_buyer_match": "#6b7280",
    "island_or_coastal_supply": "#c45c26",
    "commercial_to_3rd": "#9ca3af",
}

print("pygwalker", getattr(pyg, "__version__", "?"))
print("ROOT", ROOT)
print("AGGREGATE_BY", AGGREGATE_BY)


pygwalker 0.5.0.1
ROOT C:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\prod_nowcast
AGGREGATE_BY complex


## 1) Load delivered events

Same source you used in PyGWalker (`trades_imp_exp_master`). We keep only columns
needed for monthly product flows.

In [14]:
USE_COLS = [
    "Imp/Exp",
    "kb_imp_exp",
    "Installation",
    "Country",
    "Zone",
    "Continent",
    "Subcontinent",
    "TradingRegion",
    "Date",
    "Product",
    "Volume (bbl)",
    "Trade status",
]

if PARQUET_PATH.exists():
    raw = pd.read_parquet(PARQUET_PATH, columns=USE_COLS)
    source = PARQUET_PATH.name
else:
    assert CSV_PATH.exists(), f"Missing master under {TABLEAU_DIR}"
    raw = pd.read_csv(CSV_PATH, usecols=USE_COLS, parse_dates=["Date"])
    source = CSV_PATH.name

events = raw.loc[raw["Trade status"].eq(STATUS)].copy()
events = events.loc[
    events["Installation"].notna()
    & events["Installation"].ne("")
    & events["Product"].notna()
    & events["Product"].ne("")
    & events["Date"].notna()
]

# Absolute kb for comparable import/export; keep signed kb for FO-style charts.
events["kb_abs"] = events["Volume (bbl)"] / 1000.0
events["year_month"] = events["Date"].dt.to_period("M").dt.to_timestamp()

print(f"{source}: {len(raw):,} → {len(events):,} usable {STATUS} rows")
print("range:", events["Date"].min(), "→", events["Date"].max())

trades_imp_exp_master.parquet: 1,557,826 → 1,418,025 usable Delivered rows
range: 2016-01-02 19:09:00 → 2026-08-18 15:39:00


In [15]:
from pathlib import Path
import pandas as pd

ROOT = Path("..").resolve()  # notebook under notebooks/
# ROOT = Path(r"C:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\prod_nowcast")  # if standalone
PARQUET_PATH = ROOT / "data" / "derived" / "tableau" / "trades_imp_exp_master.parquet"
OUT_PATH = ROOT / "notebooks" / "spain_terminals_unique.csv"

COLS = [
    "Installation",
    "Country",
    "Zone",
    "Continent",
    "Subcontinent",
    "TradingRegion",
]

def mode_or_first(s: pd.Series):
    s = s.dropna()
    s = s[s.astype(str).str.strip().ne("")]
    if s.empty:
        return pd.NA
    m = s.mode()
    return m.iloc[0] if len(m) else s.iloc[0]

# Reuse `events` if already loaded in the notebook; otherwise load Delivered rows.
if "events" not in globals():
    raw = pd.read_parquet(PARQUET_PATH, columns=COLS + ["Trade status", "Date"])
    events = raw.loc[
        raw["Trade status"].eq("Delivered")
        & raw["Installation"].notna()
        & raw["Installation"].ne("")
    ].copy()

spain = events.loc[events["Country"].eq("Spain"), COLS].copy()

spain_terminals = (
    spain.groupby("Installation", dropna=False)
    .agg(
        Country=("Country", mode_or_first),
        Zone=("Zone", mode_or_first),
        Continent=("Continent", mode_or_first),
        Subcontinent=("Subcontinent", mode_or_first),
        TradingRegion=("TradingRegion", mode_or_first),
    )
    .reset_index()
    .rename(columns={"Installation": "installation_name"})
    .sort_values("installation_name")
    .reset_index(drop=True)
)

#spain_terminals.to_csv(OUT_PATH, index=False)
print(f"{len(spain_terminals)} unique Spain terminals → {OUT_PATH}")
spain_terminals.head()

69 unique Spain terminals → C:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\prod_nowcast\notebooks\spain_terminals_unique.csv


,installation_name,Country,Zone,Continent,Subcontinent,TradingRegion
0,APM Terminals Barcelona,Spain,Barcelona,Europe,Southern Europe,MED Zone (MED Sea+Black Sea)
1,ASESA,Spain,Tarragona,Europe,Southern Europe,MED Zone (MED Sea+Black Sea)
2,Alcoa San Ciprian,Spain,San Cibrao,Europe,Southern Europe,North West Europe Zone
3,Alpenor,Spain,Vigo,Europe,Southern Europe,North West Europe Zone
4,Arrecife,Spain,Arrecife,Europe,Southern Europe,Other Southern Europe


## 2) Monthly cube (chart grain)

Long format close to the master: one row per
`Installation × Product × month × Imp/Exp`.

- `kbd = kb_abs / days_in_that_month`
- `kbd_signed` / `kb_signed` mirror master sign (+ export, − import)

In [16]:
def days_in_timestamp_month(ts: pd.Timestamp) -> int:
    return calendar.monthrange(ts.year, ts.month)[1]


def mode_or_first(s: pd.Series):
    m = s.dropna()
    if m.empty:
        return pd.NA
    modes = m.mode()
    return modes.iloc[0] if len(modes) else m.iloc[0]


monthly = (
    events.groupby(
        [
            "Installation",
            "Country",
            "Zone",
            "Continent",
            "Subcontinent",
            "TradingRegion",
            "Product",
            "year_month",
            "Imp/Exp",
        ],
        dropna=False,
        observed=True,
    )
    .agg(
        kb_abs=("kb_abs", "sum"),
        kb_signed=("kb_imp_exp", "sum"),
        n_trades=("kb_abs", "size"),
    )
    .reset_index()
)

monthly["days_in_month"] = monthly["year_month"].map(days_in_timestamp_month)
monthly["kbd"] = monthly["kb_abs"] / monthly["days_in_month"]
monthly["kbd_signed"] = monthly["kb_signed"] / monthly["days_in_month"]

print(f"monthly rows: {len(monthly):,}")
monthly.head(3)

monthly rows: 451,229


,Installation,Country,Zone,Continent,Subcontinent,TradingRegion,Product,year_month,Imp/Exp,kb_abs,kb_signed,n_trades,days_in_month,kbd,kbd_signed
0,A & E Tank Farm,Nigeria,Warri,Africa,Western Africa,Western Africa,Blending Comps,2021-01-01,Import,155.031,-155.031,1,31,5.001000,-5.001000
1,A & E Tank Farm,Nigeria,Warri,Africa,Western Africa,Western Africa,Blending Comps,2024-06-01,Import,162.119,-162.119,1,30,5.403967,-5.403967
2,A & E Tank Farm,Nigeria,Warri,Africa,Western Africa,Western Africa,Diesel,2024-05-01,Import,238.735,-238.735,2,31,7.701129,-7.701129


## 3) Lifetime Installation × Product roles

| role | rule |
|---|---|
| `demand_center` | `import_share ≥ 0.70` and `total_kb ≥ MIN_TOTAL_KB` |
| `supply_center` | `import_share ≤ 0.30` and `total_kb ≥ MIN_TOTAL_KB` |
| `balanced_hub` | in between and above the volume floor |
| `thin` | below `MIN_TOTAL_KB` |

`import_kbd_avg` / `export_kbd_avg` = mean monthly `kbd` for that direction
(averaged over months where that Installation×Product×direction had volume).

In [17]:
def classify_role(import_share: float, total_kb: float) -> str:
    if total_kb < MIN_TOTAL_KB or pd.isna(import_share):
        return "thin"
    if import_share >= IMPORT_HEAVY_MIN:
        return "demand_center"
    if import_share <= EXPORT_HEAVY_MAX:
        return "supply_center"
    return "balanced_hub"


# Stable geo attrs once per installation (avoids splitting roles on rare geo drift).
geo = events.groupby("Installation", dropna=False).agg(
    Country=("Country", mode_or_first),
    Zone=("Zone", mode_or_first),
    Continent=("Continent", mode_or_first),
    Subcontinent=("Subcontinent", mode_or_first),
    TradingRegion=("TradingRegion", mode_or_first),
)

dir_stats = (
    monthly.groupby(["Installation", "Product", "Imp/Exp"], dropna=False)
    .agg(
        kb=("kb_abs", "sum"),
        kbd_avg=("kbd", "mean"),
        n_months=("year_month", "nunique"),
        first_month=("year_month", "min"),
        last_month=("year_month", "max"),
    )
    .reset_index()
)

wide = dir_stats.pivot_table(
    index=["Installation", "Product"],
    columns="Imp/Exp",
    values=["kb", "kbd_avg", "n_months", "first_month", "last_month"],
    aggfunc="first",
)
wide.columns = [f"{metric}_{direction.lower()}" for metric, direction in wide.columns]
roles = wide.reset_index()

for c, default in [
    ("kb_import", 0.0),
    ("kb_export", 0.0),
    ("kbd_avg_import", 0.0),
    ("kbd_avg_export", 0.0),
    ("n_months_import", 0),
    ("n_months_export", 0),
]:
    if c not in roles.columns:
        roles[c] = default
    else:
        roles[c] = roles[c].fillna(default)

roles = roles.rename(
    columns={
        "kb_import": "import_kb",
        "kb_export": "export_kb",
        "kbd_avg_import": "import_kbd_avg",
        "kbd_avg_export": "export_kbd_avg",
    }
)

roles["total_kb"] = roles["import_kb"] + roles["export_kb"]
roles["net_kb"] = roles["import_kb"] - roles["export_kb"]
roles["import_share"] = roles["import_kb"] / roles["total_kb"].replace(0, pd.NA)
roles["total_kbd_avg"] = roles["import_kbd_avg"].fillna(0) + roles["export_kbd_avg"].fillna(0)
roles["n_months"] = roles[["n_months_import", "n_months_export"]].max(axis=1)
roles["first_month"] = roles[[c for c in roles.columns if c.startswith("first_month_")]].min(axis=1)
roles["last_month"] = roles[[c for c in roles.columns if c.startswith("last_month_")]].max(axis=1)
roles["role"] = [
    classify_role(s, t) for s, t in zip(roles["import_share"], roles["total_kb"])
]

term_total = roles.groupby("Installation")["total_kb"].transform("sum")
roles["product_share_of_terminal"] = roles["total_kb"] / term_total.replace(0, pd.NA)

roles = roles.merge(geo, left_on="Installation", right_index=True, how="left")

print(roles["role"].value_counts())
print(f"role rows: {len(roles):,}")

role
thin             8787
demand_center    5420
supply_center    2381
balanced_hub     1219
Name: count, dtype: int64
role rows: 17,807


## 4) Complex map + working grain (`AGGREGATE_BY`)

Manual desk map: `data/manual/terminal_desk_map.csv`.

Current mapped complexes include **Algeciras Storage**, **Bilbao Ref+Port**,
**Barcelona Port** (Vopak Terquimsa + 32A–32G). **APM Terminals Barcelona** stays
out — airport Jet for El Prat, not the port/storage pool.

Unmapped installations become **singleton** complexes (`inst:<name>`) so the
complex view covers the whole country.

Set `AGGREGATE_BY` in the setup cell:
- `"complex"` — typology on economic units (recommended for your iteration)
- `"installation"` — raw Kpler sites (map validation / drill-down)


In [18]:
desk = load_desk_map(DESK_MAP_PATH)
cmap = complex_map_view(desk)

# Every installation -> entity (desk complex or singleton)
entity_map = (
    roles[["Installation", "Country", "Zone"]]
    .drop_duplicates(subset=["Installation"])
    .merge(cmap, on="Installation", how="left")
)
desk_mapped = entity_map["complex_id"].notna()
entity_map["is_desk_complex"] = desk_mapped
entity_map.loc[~desk_mapped, "complex_id"] = "inst:" + entity_map.loc[~desk_mapped, "Installation"]
entity_map.loc[~desk_mapped, "complex_name"] = entity_map.loc[~desk_mapped, "Installation"]

# Attach desk/singleton ids onto installation roles + monthly
roles = roles.drop(columns=[c for c in ("complex_id", "complex_name", "is_desk_complex") if c in roles.columns])
roles = roles.merge(
    entity_map[["Installation", "complex_id", "complex_name", "is_desk_complex"]],
    on="Installation",
    how="left",
)
monthly = monthly.drop(columns=[c for c in ("complex_id", "complex_name", "is_desk_complex") if c in monthly.columns])
monthly = monthly.merge(
    entity_map[["Installation", "complex_id", "complex_name", "is_desk_complex"]],
    on="Installation",
    how="left",
)

# Complex-month cube (desk + singletons): sum member kb/kbd inside the month
complex_monthly = (
    monthly.groupby(
        ["complex_id", "complex_name", "Product", "year_month", "Imp/Exp"],
        dropna=False,
    )
    .agg(kb_abs=("kb_abs", "sum"), kb_signed=("kb_signed", "sum"), kbd=("kbd", "sum"), kbd_signed=("kbd_signed", "sum"), n_trades=("n_trades", "sum"))
    .reset_index()
)

# Geo for a complex = mode across member events (via installation geo already on monthly)
complex_geo = (
    monthly.groupby("complex_id", dropna=False)
    .agg(
        Country=("Country", mode_or_first),
        Zone=("Zone", mode_or_first),
        Continent=("Continent", mode_or_first),
        Subcontinent=("Subcontinent", mode_or_first),
        TradingRegion=("TradingRegion", mode_or_first),
        n_installations=("Installation", "nunique"),
        is_desk_complex=("is_desk_complex", "max"),
    )
    .reset_index()
)

c_dir = (
    complex_monthly.groupby(["complex_id", "complex_name", "Product", "Imp/Exp"], dropna=False)
    .agg(
        kb=("kb_abs", "sum"),
        kbd_avg=("kbd", "mean"),
        n_months=("year_month", "nunique"),
        first_month=("year_month", "min"),
        last_month=("year_month", "max"),
    )
    .reset_index()
)

c_wide = c_dir.pivot_table(
    index=["complex_id", "complex_name", "Product"],
    columns="Imp/Exp",
    values=["kb", "kbd_avg", "n_months", "first_month", "last_month"],
    aggfunc="first",
)
c_wide.columns = [f"{metric}_{direction.lower()}" for metric, direction in c_wide.columns]
complex_roles = c_wide.reset_index()

for c, default in [
    ("kb_import", 0.0),
    ("kb_export", 0.0),
    ("kbd_avg_import", 0.0),
    ("kbd_avg_export", 0.0),
    ("n_months_import", 0),
    ("n_months_export", 0),
]:
    if c not in complex_roles.columns:
        complex_roles[c] = default
    else:
        complex_roles[c] = complex_roles[c].fillna(default)

complex_roles = complex_roles.rename(
    columns={
        "kb_import": "import_kb",
        "kb_export": "export_kb",
        "kbd_avg_import": "import_kbd_avg",
        "kbd_avg_export": "export_kbd_avg",
    }
)
complex_roles["total_kb"] = complex_roles["import_kb"] + complex_roles["export_kb"]
complex_roles["net_kb"] = complex_roles["import_kb"] - complex_roles["export_kb"]
complex_roles["import_share"] = complex_roles["import_kb"] / complex_roles["total_kb"].replace(0, pd.NA)
complex_roles["total_kbd_avg"] = (
    complex_roles["import_kbd_avg"].fillna(0) + complex_roles["export_kbd_avg"].fillna(0)
)
complex_roles["n_months"] = complex_roles[["n_months_import", "n_months_export"]].max(axis=1)
complex_roles["first_month"] = complex_roles[[c for c in complex_roles.columns if c.startswith("first_month_")]].min(axis=1)
complex_roles["last_month"] = complex_roles[[c for c in complex_roles.columns if c.startswith("last_month_")]].max(axis=1)
complex_roles["role"] = [
    classify_role(s, t) for s, t in zip(complex_roles["import_share"], complex_roles["total_kb"])
]
cx_total = complex_roles.groupby("complex_id")["total_kb"].transform("sum")
complex_roles["product_share_of_terminal"] = complex_roles["total_kb"] / cx_total.replace(0, pd.NA)
complex_roles = complex_roles.merge(complex_geo, on="complex_id", how="left")

# --- Working views driven by AGGREGATE_BY ---
if AGGREGATE_BY == "complex":
    roles_view = complex_roles.rename(
        columns={"complex_id": "entity_id", "complex_name": "entity_name"}
    ).copy()
    monthly_view = complex_monthly.rename(
        columns={"complex_id": "entity_id", "complex_name": "entity_name"}
    ).copy()
    # country filter helper on monthly_view via complex_geo
    monthly_view = monthly_view.merge(
        complex_geo[["complex_id", "Country", "Zone"]].rename(columns={"complex_id": "entity_id"}),
        on="entity_id",
        how="left",
    )
else:
    roles_view = roles.rename(columns={"Installation": "entity_name"}).copy()
    roles_view["entity_id"] = roles_view["entity_name"]
    monthly_view = monthly.rename(columns={"Installation": "entity_name"}).copy()
    monthly_view["entity_id"] = monthly_view["entity_name"]

print(f"desk-mapped installations: {int(entity_map['is_desk_complex'].sum())}")
print(f"desk complexes: {entity_map.loc[entity_map['is_desk_complex'], 'complex_name'].nunique()}")
print(f"complex_roles rows (incl. singletons): {len(complex_roles):,}")
print(f"roles_view grain: {AGGREGATE_BY} | rows: {len(roles_view):,}")
display(
    entity_map.loc[entity_map["is_desk_complex"]]
    .groupby(["complex_id", "complex_name"], dropna=False)
    .agg(n=("Installation", "nunique"), members=("Installation", lambda s: ", ".join(sorted(s.astype(str)))))
    .reset_index()
)

# Member -> complex impact (desk complexes only) — core of the iterative loop
desk_ids = set(entity_map.loc[entity_map["is_desk_complex"], "complex_id"])
impact = (
    roles.loc[roles["complex_id"].isin(desk_ids)]
    [
        [
            "complex_name",
            "Installation",
            "Product",
            "role",
            "import_share",
            "import_kb",
            "export_kb",
            "total_kb",
            "import_kbd_avg",
            "export_kbd_avg",
        ]
    ]
    .rename(
        columns={
            "role": "member_role",
            "import_share": "member_import_share",
            "import_kb": "member_import_kb",
            "export_kb": "member_export_kb",
            "total_kb": "member_total_kb",
            "import_kbd_avg": "member_import_kbd_avg",
            "export_kbd_avg": "member_export_kbd_avg",
        }
    )
    .merge(
        complex_roles.loc[complex_roles["complex_id"].isin(desk_ids)][
            [
                "complex_name",
                "Product",
                "role",
                "import_share",
                "import_kb",
                "export_kb",
                "total_kb",
                "import_kbd_avg",
                "export_kbd_avg",
            ]
        ].rename(
            columns={
                "role": "complex_role",
                "import_share": "complex_import_share",
                "import_kb": "complex_import_kb",
                "export_kb": "complex_export_kb",
                "total_kb": "complex_total_kb",
                "import_kbd_avg": "complex_import_kbd_avg",
                "export_kbd_avg": "complex_export_kbd_avg",
            }
        ),
        on=["complex_name", "Product"],
        how="left",
    )
)
impact["role_changed"] = impact["member_role"] != impact["complex_role"]

print("\nMember → complex impact (desk complexes; role_changed highlighted conceptually):")
display(
    impact.sort_values(["complex_name", "Product", "member_total_kb"], ascending=[True, True, False])[
        [
            "complex_name",
            "Product",
            "Installation",
            "member_role",
            "complex_role",
            "role_changed",
            "member_import_share",
            "complex_import_share",
            "member_total_kb",
            "complex_total_kb",
        ]
    ]
)

print("\nExample desk complexes — complex-level roles:")
display(
    complex_roles.loc[
        complex_roles["complex_name"].isin(EXAMPLE_COMPLEXES) & complex_roles["is_desk_complex"].fillna(False)
    ]
    .sort_values(["complex_name", "total_kb"], ascending=[True, False])[
        [
            "complex_name",
            "Product",
            "role",
            "import_share",
            "import_kbd_avg",
            "export_kbd_avg",
            "total_kb",
            "n_installations",
        ]
    ]
)


desk-mapped installations: 61
desk complexes: 25
complex_roles rows (incl. singletons): 17,587
roles_view grain: complex | rows: 17,587


,complex_id,complex_name,n,members
0,SIOT_storage,NaN,2,"Teseco Trieste, Trieste"
1,algeciras_storage,Algeciras Storage,2,"CLH Algeciras Storage, Vopak Algeciras"
2,aviles_smelter+port,Asturiana Del Zinc,3,"Asturiana Del Zinc, Aviles, Valliniello"
3,barcelona_port_storage,Barcelona Port,7,"Barcelona 32A, Barcelona 32B, Barcelona 32C, B..."
4,bilbao_port,Bilbao refinery & port,2,"Bilbao Oil Facilities, Repsol Petronor Refinery"
5,cartagena_ref+port,Cartagena refinery & port,3,"ICL Fuentes Fertilizantes, Repsol Cartagena Re..."
6,castellon_refinery,BP Castellon Refinery,2,"BP Castellon Refinery, Masol Iberia Castellon"
7,chane_le_havre_storage,Chane Le Havre Storage,3,"Chane Le Havre 1, Chane Le Havre 2, Chane Sotr..."
8,coruna_ref+port,Repsol Coruna Refinery,1,Repsol Coruna Refinery
9,donges_ref,Donges refinery,3,"Cargill & Yara Montoir, Cargill Saint-Nazaire,..."



Member → complex impact (desk complexes; role_changed highlighted conceptually):


,complex_name,Product,Installation,member_role,complex_role,role_changed,member_import_share,complex_import_share,member_total_kb,complex_total_kb
97,Algeciras Storage,Bitumen/Asphalt,CLH Algeciras Storage,thin,thin,False,0.715138,0.661062,257.233,295.467
469,Algeciras Storage,Bitumen/Asphalt,Vopak Algeciras,thin,thin,False,0.297249,0.661062,38.234,295.467
98,Algeciras Storage,Blending Comps,CLH Algeciras Storage,thin,thin,False,1.0,1.0,240.652,444.270
470,Algeciras Storage,Blending Comps,Vopak Algeciras,thin,thin,False,1.0,1.0,203.618,444.270
471,Algeciras Storage,CBFS,Vopak Algeciras,balanced_hub,balanced_hub,False,0.348505,0.328118,540.044,573.599
...,...,...,...,...,...,...,...,...,...,...
466,NaN,Gasoline,Trieste,thin,NaN,True,0.348007,NaN,152.991,NaN
441,NaN,Jet,Teseco Trieste,thin,NaN,True,1.0,NaN,101.900,NaN
442,NaN,Kerosene,Teseco Trieste,thin,NaN,True,1.0,NaN,87.893,NaN
443,NaN,Naphtha,Teseco Trieste,thin,NaN,True,1.0,NaN,123.991,NaN



Example desk complexes — complex-level roles:


,complex_name,Product,role,import_share,import_kbd_avg,export_kbd_avg,total_kb,n_installations
7,Algeciras Storage,FO,demand_center,0.854435,55.250585,10.183091,229762.373,2
6,Algeciras Storage,Dirty Feedstocks,demand_center,0.944921,11.933394,3.088238,30793.517,2
8,Algeciras Storage,Gasoil,demand_center,0.856821,8.754684,4.604232,30474.582,2
13,Algeciras Storage,SRFO,demand_center,0.987222,6.400345,1.186301,8532.163,2
5,Algeciras Storage,Diesel,demand_center,1.0,4.723467,0.000000,3015.430,2
9,Algeciras Storage,Gasoline,demand_center,0.753566,3.911959,2.211078,2984.601,2
14,Algeciras Storage,Slurry,demand_center,0.969029,3.438675,1.292645,1293.859,2
12,Algeciras Storage,Naphtha,demand_center,0.982916,3.538947,0.661226,1199.852,2
4,Algeciras Storage,Cutter Stock,demand_center,0.971779,1.866472,0.604200,642.279,2
2,Algeciras Storage,CBFS,balanced_hub,0.328118,1.581797,1.407527,573.599,2


## 5) Understanding the output (all products)

Working tables follow `AGGREGATE_BY` from the setup cell.

| Table | Grain | Use it to… |
|---|---|---|
| `monthly` / `monthly_view` | install or complex × product × month × Imp/Exp | time charts |
| `roles` | installation × product | berth-level labels |
| `complex_roles` | complex × product (desk + singletons) | economic-unit labels |
| `roles_view` | selected grain | briefing charts / ranked centers |
| `impact` | member vs desk complex | see what aggregation changed |

**How to read a role row**
- `import_share` high → inbound-heavy (`demand_center` if large enough)
- `*_kbd_avg` → typical monthly flow rate
- `thin` → below `MIN_TOTAL_KB`

APM Barcelona is intentionally **not** in Barcelona Port (El Prat Jet). Use the
impact table + Zone review list when iterating membership.


In [19]:
focus = roles_view.loc[roles_view["Country"].eq(FOCUS_COUNTRY)].copy()
focus_labeled = focus.loc[focus["role"].ne("thin")].copy()
entity_col = "entity_name"

print(f"AGGREGATE_BY={AGGREGATE_BY} | {FOCUS_COUNTRY}")
print(f"entities: {focus[entity_col].nunique()} | rows: {len(focus):,} | non-thin: {len(focus_labeled):,}")
print("role mix (rows):")
print(focus["role"].value_counts())
print("\nrole mix by product (rows):")
display(
    focus.pivot_table(index="Product", columns="role", values=entity_col, aggfunc="count", fill_value=0)
    .reindex(columns=["demand_center", "supply_center", "balanced_hub", "thin"], fill_value=0)
    .sort_index()
)

ranked_parts = []
for role_name in ("demand_center", "supply_center"):
    metric = "import_kbd_avg" if role_name == "demand_center" else "export_kbd_avg"
    chunk = (
        focus.loc[focus["role"].eq(role_name)]
        .sort_values(["Product", metric], ascending=[True, False])
        .groupby("Product", group_keys=False)
        .head(TOP_N_PER_PRODUCT)
        .assign(rank_within_product=lambda d: d.groupby("Product").cumcount() + 1)
    )
    ranked_parts.append(chunk)

ranked = pd.concat(ranked_parts, ignore_index=True)
print(f"\nTop {TOP_N_PER_PRODUCT} {FOCUS_COUNTRY} centers per product ({AGGREGATE_BY} grain):")
cols = [
    "role",
    "Product",
    "rank_within_product",
    entity_col,
    "Zone",
    "import_share",
    "import_kbd_avg",
    "export_kbd_avg",
    "total_kb",
    "product_share_of_terminal",
]
if "n_installations" in ranked.columns:
    cols.insert(5, "n_installations")
display(ranked[[c for c in cols if c in ranked.columns]].sort_values(["role", "Product", "rank_within_product"]))

# Zone review: same-Zone sites not in a desk complex (membership candidates)
if AGGREGATE_BY == "complex":
    desk_zones = (
        entity_map.loc[entity_map["is_desk_complex"] & entity_map["Country"].eq(FOCUS_COUNTRY), "Zone"]
        .dropna()
        .unique()
    )
    candidates = entity_map.loc[
        entity_map["Country"].eq(FOCUS_COUNTRY)
        & entity_map["Zone"].isin(desk_zones)
        & ~entity_map["is_desk_complex"]
    ][["Installation", "Zone", "complex_name"]]
    print("\nSame-Zone sites NOT in a desk complex (review only — not auto-merged):")
    display(candidates.sort_values(["Zone", "Installation"]))

print("\nDesk-complex impact summary (role changes):")
if len(impact):
    chg = (
        impact.loc[impact["role_changed"]]
        .groupby(["complex_name", "Product"], dropna=False)
        .agg(
            n_members_changed=("Installation", "nunique"),
            complex_role=("complex_role", "first"),
            complex_import_share=("complex_import_share", "first"),
        )
        .reset_index()
        .sort_values(["complex_name", "Product"])
    )
    display(chg)
else:
    print("(no impact rows)")


AGGREGATE_BY=complex | Germany
entities: 37 | rows: 229 | non-thin: 79
role mix (rows):
role
thin             150
demand_center     41
supply_center     20
balanced_hub      18
Name: count, dtype: int64

role mix by product (rows):


role,demand_center,supply_center,balanced_hub,thin
Product,,,,
Bitumen/Asphalt,1,3,0,5
Blending Comps,4,1,3,17
CBFS,0,1,0,4
Clean Condensate,1,0,0,5
Coal Tar,0,0,0,4
Cutter Stock,0,0,0,2
Diesel,10,3,0,13
Dirty Feedstocks,3,3,2,6
FO,5,3,2,12



Top 5 Germany centers per product (complex grain):


,role,Product,rank_within_product,entity_name,Zone,n_installations,import_share,import_kbd_avg,export_kbd_avg,total_kb,product_share_of_terminal
0,demand_center,Bitumen/Asphalt,1,Blexen Tank Farm,Nordenham,1,0.775383,1.210619,1.042087,2952.734,0.965774
1,demand_center,Blending Comps,1,Oiltank. Hamburg,Hamburg,1,0.843555,2.344570,1.720452,681.823,0.003666
2,demand_center,Blending Comps,2,BMÔ Bremen,Bremen,1,0.896952,1.099239,1.846836,1645.740,0.025088
3,demand_center,Blending Comps,3,Evos Hamburg,Hamburg,1,0.791505,0.969292,1.777653,749.110,0.005198
4,demand_center,Blending Comps,4,Wilhelmshaven Refinery,Wilhelmshaven,2,1.0,0.965723,0.000000,782.331,0.006819
5,demand_center,Clean Condensate,1,NWO Tank Farm,Wilhelmshaven,1,1.0,7.451079,0.000000,2291.717,0.143721
6,demand_center,Diesel,1,Oiltank. Hamburg,Hamburg,1,0.998513,38.733304,3.355424,137075.757,0.736993
7,demand_center,Diesel,2,Evos Hamburg,Hamburg,1,0.9928,25.321407,10.479210,90236.732,0.626133
8,demand_center,Diesel,3,BMÔ Bremen,Bremen,1,0.999541,15.610184,0.802367,52480.725,0.800024
9,demand_center,Diesel,4,NWO Tank Farm,Wilhelmshaven,1,1.0,13.690657,0.000000,1264.765,0.079318



Same-Zone sites NOT in a desk complex (review only — not auto-merged):


,Installation,Zone,complex_name
2087,Power Oil Rostock,Rostock,Power Oil Rostock
1660,NWO Tank Farm,Wilhelmshaven,NWO Tank Farm



Desk-complex impact summary (role changes):


,complex_name,Product,n_members_changed,complex_role,complex_import_share
0,Algeciras Storage,CBFS,1,balanced_hub,0.328118
1,Algeciras Storage,Cutter Stock,1,demand_center,0.971779
2,Algeciras Storage,Naphtha,1,demand_center,0.982916
3,Algeciras Storage,Slurry,1,demand_center,0.969029
4,BP Castellon Refinery,Blending Comps,1,demand_center,0.864975
...,...,...,...,...,...
103,NaN,Gasoline,2,None,None
104,NaN,Jet,1,None,None
105,NaN,Kerosene,1,None,None
106,NaN,Naphtha,1,None,None


### Charts — country briefing (all products)

1. **Role mix by product** — where demand vs supply labels concentrate  
2. **Import vs export kbd scatter** — each point is one Installation × Product  
3. **Top centers per product** — demand (import kbd) and supply (export kbd)  
4. **Example installations heatmap** — `import_share` across products (Algeciras path)

In [20]:
# --- 1) Role mix by product (volume-weighted, clearer than row counts) ---
mix = (
    focus.groupby(["Product", "role"], dropna=False)["total_kb"]
    .sum()
    .reset_index()
)
fig_mix = px.bar(
    mix,
    x="Product",
    y="total_kb",
    color="role",
    color_discrete_map=ROLE_COLORS,
    category_orders={"role": ["demand_center", "supply_center", "balanced_hub", "thin"]},
    title=f"{FOCUS_COUNTRY}: lifetime kb by product and role",
    labels={"total_kb": "total kb", "Product": "product"},
)
fig_mix.update_layout(xaxis_tickangle=-35, legend_title_text="role", height=420)
fig_mix.show()

# --- 2) Demand / supply map: every non-thin Installation × Product ---
# hover fields that exist on roles_view (complex grain has entity_name, not complex_name)
_scatter_hover = {
    "Product": True,
    "import_share": ":.2f",
    "Zone": True,
    "total_kb": ":.0f",
    "export_kbd_avg": ":.2f",
    "import_kbd_avg": ":.2f",
    "role": True,
}
if "n_installations" in focus_labeled.columns:
    _scatter_hover["n_installations"] = True
if "is_desk_complex" in focus_labeled.columns:
    _scatter_hover["is_desk_complex"] = True

fig_scatter = px.scatter(
    focus_labeled,
    x="export_kbd_avg",
    y="import_kbd_avg",
    color="role",
    size="total_kb",
    size_max=36,
    hover_name=entity_col,
    hover_data=_scatter_hover,
    color_discrete_map=ROLE_COLORS,
    title=f"{FOCUS_COUNTRY}: import vs export kbd (point = {AGGREGATE_BY} × Product)",
    labels={
        "export_kbd_avg": "export kbd (avg month)",
        "import_kbd_avg": "import kbd (avg month)",
    },
)
fig_scatter.update_layout(height=520)
fig_scatter.show()

# --- 3) Where demand / supply centers sit, by product (treemap) ---
demand = focus.loc[focus["role"].eq("demand_center")].copy()
supply = focus.loc[focus["role"].eq("supply_center")].copy()

if len(demand):
    fig_demand = px.treemap(
        demand,
        path=["Product", entity_col],
        values="import_kbd_avg",
        color="import_share",
        color_continuous_scale="Oranges",
        title=f"{FOCUS_COUNTRY}: demand centers ({AGGREGATE_BY}; size = import kbd avg)",
        hover_data={"total_kb": ":.0f", "Zone": True},
    )
    fig_demand.update_layout(height=560, margin=dict(t=50, l=10, r=10, b=10))
    fig_demand.show()
else:
    print("No demand_center rows in focus country.")

if len(supply):
    fig_supply = px.treemap(
        supply,
        path=["Product", entity_col],
        values="export_kbd_avg",
        color="import_share",
        color_continuous_scale="Blues_r",
        title=f"{FOCUS_COUNTRY}: supply centers ({AGGREGATE_BY}; size = export kbd avg)",
        hover_data={"total_kb": ":.0f", "Zone": True},
    )
    fig_supply.update_layout(height=560, margin=dict(t=50, l=10, r=10, b=10))
    fig_supply.show()
else:
    print("No supply_center rows in focus country.")

# --- 4) Example entities: import_share heatmap across products ---
example_names = EXAMPLE_COMPLEXES if AGGREGATE_BY == "complex" else EXAMPLE_INSTALLATIONS
ex = focus.loc[focus[entity_col].isin(example_names)].copy()
if ex.empty:
    print("Example entities not found in focus country — skip heatmap.")
else:
    heat = ex.pivot_table(
        index=entity_col,
        columns="Product",
        values="import_share",
        aggfunc="first",
    )
    prod_order = (
        ex.groupby("Product")["total_kb"].sum().sort_values(ascending=False).index.tolist()
    )
    heat = heat.reindex(columns=prod_order)

    fig_heat = px.imshow(
        heat,
        color_continuous_scale="RdYlBu_r",
        zmin=0,
        zmax=1,
        aspect="auto",
        title=(
            f"Example {AGGREGATE_BY} entities — import_share by product "
            f"(≥{IMPORT_HEAVY_MIN:.0%} ≈ demand, ≤{EXPORT_HEAVY_MAX:.0%} ≈ supply)"
        ),
        labels={"color": "import_share"},
    )
    fig_heat.update_layout(height=320 + 40 * len(heat), xaxis_tickangle=-35)
    fig_heat.show()

    role_mat = ex.pivot_table(
        index=entity_col, columns="Product", values="role", aggfunc="first"
    ).reindex(columns=prod_order)
    print("Role tags for the same example matrix:")
    display(role_mat)


Example entities not found in focus country — skip heatmap.


In [21]:
# Optional drill-down: set AGGREGATE_BY = "installation" and re-run from setup,
# or filter impact/roles for a single complex_name.
print("Desk complexes in map:", sorted(entity_map.loc[entity_map["is_desk_complex"], "complex_name"].unique()))
print("APM in Barcelona Port?", "APM Terminals Barcelona" in set(
    entity_map.loc[entity_map["complex_name"].eq("Barcelona Port"), "Installation"]
))


TypeError: '<' not supported between instances of 'float' and 'str'

## 6) Pattern monitor — predictability & soft demand/supply

Lifetime `roles` answer “what kind of center is this?”. This section asks:

1. **Is the Imp/Exp pattern historically regular?** (CV, weak/strong YoY seasonality, rolling role stability)
2. **Is the last 3 months soft vs history?** for import (demand proxy) and export (supply proxy)

Soft flags use **both**:
- `z` vs all prior months (level shock)
- `seasonal_ratio` = recent 3m mean / mean of the **same calendar months** in prior years  
  (so a quiet winter is not flagged just because summers are busy)

Requires `MIN_PATTERN_MONTHS` of history. Missing months inside a series span are treated as 0 kbd (no cargo that month).

In [22]:
def _complete_monthly_kbd(year_month: pd.Series, values: pd.Series) -> pd.Series:
    """Month-start index from first to last activity; gaps -> 0 kbd."""
    tmp = pd.DataFrame({"year_month": year_month, "kbd": values}).sort_values("year_month")
    tmp = tmp.groupby("year_month", as_index=False)["kbd"].sum()
    idx = pd.date_range(tmp["year_month"].min(), tmp["year_month"].max(), freq="MS")
    s = tmp.set_index("year_month")["kbd"].reindex(idx, fill_value=0.0)
    s.index.name = "year_month"
    return s


def _direction_softness(s: pd.Series) -> dict:
    """Last RECENT_MONTHS vs prior history + same-calendar-month baseline."""
    if len(s) < MIN_PATTERN_MONTHS:
        return {
            "recent_kbd": np.nan,
            "hist_mean": np.nan,
            "z": np.nan,
            "pctile": np.nan,
            "seasonal_expected": np.nan,
            "seasonal_ratio": np.nan,
            "soft": False,
        }

    recent = s.iloc[-RECENT_MONTHS:]
    hist = s.iloc[:-RECENT_MONTHS]
    recent_kbd = float(recent.mean())
    hist_mean = float(hist.mean())
    hist_std = float(hist.std(ddof=0))
    z = (recent_kbd - hist_mean) / hist_std if hist_std > 1e-9 else np.nan
    pctile = float((hist <= recent_kbd).mean()) if len(hist) else np.nan

    # Same calendar months in prior years (seasonality-aware baseline)
    seasonal_vals = []
    for ts, val in recent.items():
        prior_same = hist[hist.index.month == ts.month]
        if len(prior_same):
            seasonal_vals.append(float(prior_same.mean()))
    seasonal_expected = float(np.mean(seasonal_vals)) if seasonal_vals else np.nan
    seasonal_ratio = (
        recent_kbd / seasonal_expected
        if pd.notna(seasonal_expected) and seasonal_expected > 1e-9
        else np.nan
    )

    soft = False
    if recent_kbd >= 0 and hist_mean >= MIN_SOFT_MEAN_KBD:
        z_soft = pd.notna(z) and z <= SOFT_Z
        seas_soft = pd.notna(seasonal_ratio) and seasonal_ratio <= SOFT_SEASONAL_RATIO
        soft = bool(z_soft or seas_soft)

    return {
        "recent_kbd": recent_kbd,
        "hist_mean": hist_mean,
        "z": z,
        "pctile": pctile,
        "seasonal_expected": seasonal_expected,
        "seasonal_ratio": seasonal_ratio,
        "soft": soft,
    }


def _yoy_month_corr(s: pd.Series) -> float:
    """Mean corr of each calendar month with the prior year (seasonal predictability)."""
    corrs = []
    df = s.rename("v").reset_index()
    df["month"] = df["year_month"].dt.month
    for _, gm in df.groupby("month"):
        gm = gm.sort_values("year_month")
        if len(gm) < 3:
            continue
        with np.errstate(invalid="ignore", divide="ignore"):
            c = gm["v"].corr(gm["v"].shift(1))
        if pd.notna(c):
            corrs.append(float(c))
    return float(np.mean(corrs)) if corrs else np.nan


def pattern_stats_for_group(g: pd.DataFrame) -> pd.Series:
    """One Installation×Product (or complex×Product) monthly long slice -> pattern row."""
    # Aggregate Imp/Exp within month if duplicate geo rows slipped in
    wide = (
        g.groupby(["year_month", "Imp/Exp"], dropna=False)["kbd"]
        .sum()
        .unstack(fill_value=0.0)
        .reset_index()
    )
    for col in ("Import", "Export"):
        if col not in wide.columns:
            wide[col] = 0.0
    imp = _complete_monthly_kbd(wide["year_month"], wide["Import"])
    exp = _complete_monthly_kbd(wide["year_month"], wide["Export"]).reindex(
        imp.index, fill_value=0.0
    )
    total = imp + exp

    empty = {
        "n_months": int(len(imp)),
        "enough_history": False,
        "import_cv": np.nan,
        "export_cv": np.nan,
        "import_yoy_corr": np.nan,
        "export_yoy_corr": np.nan,
        "import_recent_kbd": np.nan,
        "export_recent_kbd": np.nan,
        "import_hist_mean_kbd": np.nan,
        "export_hist_mean_kbd": np.nan,
        "import_z": np.nan,
        "export_z": np.nan,
        "import_pctile": np.nan,
        "export_pctile": np.nan,
        "import_seasonal_expected": np.nan,
        "export_seasonal_expected": np.nan,
        "import_seasonal_ratio": np.nan,
        "export_seasonal_ratio": np.nan,
        "import_soft": False,
        "export_soft": False,
        "roll_import_share_mean": np.nan,
        "roll_import_share_std": np.nan,
        "role_stability": np.nan,
        "asof_month": imp.index.max() if len(imp) else pd.NaT,
    }
    if len(imp) < MIN_PATTERN_MONTHS:
        return pd.Series(empty)

    imp_soft = _direction_softness(imp)
    exp_soft = _direction_softness(exp)

    share = (imp / total.replace(0, np.nan)).fillna(0.5)
    roll = share.rolling(ROLLING_ROLE_MONTHS, min_periods=ROLLING_ROLE_MONTHS)
    roll_mean = float(roll.mean().iloc[-1]) if roll.mean().notna().any() else np.nan
    roll_std = float(roll.std(ddof=0).iloc[-1]) if roll.std(ddof=0).notna().any() else np.nan
    # Stability: share of last ROLLING months that agree with lifetime-style bucket of roll_mean
    recent_share = share.iloc[-ROLLING_ROLE_MONTHS:]
    if pd.isna(roll_mean):
        role_stability = np.nan
    elif roll_mean >= IMPORT_HEAVY_MIN:
        role_stability = float((recent_share >= IMPORT_HEAVY_MIN).mean())
    elif roll_mean <= EXPORT_HEAVY_MAX:
        role_stability = float((recent_share <= EXPORT_HEAVY_MAX).mean())
    else:
        role_stability = float(
            ((recent_share > EXPORT_HEAVY_MAX) & (recent_share < IMPORT_HEAVY_MIN)).mean()
        )

    def _cv(x: pd.Series) -> float:
        mu = float(x.mean())
        return float(x.std(ddof=0) / mu) if mu > 1e-9 else np.nan

    return pd.Series(
        {
            "n_months": int(len(imp)),
            "enough_history": True,
            "import_cv": _cv(imp),
            "export_cv": _cv(exp),
            "import_yoy_corr": _yoy_month_corr(imp),
            "export_yoy_corr": _yoy_month_corr(exp),
            "import_recent_kbd": imp_soft["recent_kbd"],
            "export_recent_kbd": exp_soft["recent_kbd"],
            "import_hist_mean_kbd": imp_soft["hist_mean"],
            "export_hist_mean_kbd": exp_soft["hist_mean"],
            "import_z": imp_soft["z"],
            "export_z": exp_soft["z"],
            "import_pctile": imp_soft["pctile"],
            "export_pctile": exp_soft["pctile"],
            "import_seasonal_expected": imp_soft["seasonal_expected"],
            "export_seasonal_expected": exp_soft["seasonal_expected"],
            "import_seasonal_ratio": imp_soft["seasonal_ratio"],
            "export_seasonal_ratio": exp_soft["seasonal_ratio"],
            "import_soft": imp_soft["soft"],
            "export_soft": exp_soft["soft"],
            "roll_import_share_mean": roll_mean,
            "roll_import_share_std": roll_std,
            "role_stability": role_stability,
            "asof_month": imp.index.max(),
        }
    )

# Build patterns on the working grain (installation or complex).
monthly_focus = monthly_view.loc[monthly_view["Country"].eq(FOCUS_COUNTRY)].copy()
# pattern_stats_for_group expects Imp/Exp + kbd + year_month
pattern_rows = []
for (entity_id, entity_name, product), g in monthly_focus.groupby(
    ["entity_id", "entity_name", "Product"], dropna=False
):
    row = pattern_stats_for_group(g)
    row["entity_id"] = entity_id
    row["entity_name"] = entity_name
    row["Product"] = product
    pattern_rows.append(row)
patterns = pd.DataFrame(pattern_rows)

patterns = patterns.merge(
    roles_view[
        [
            c
            for c in [
                "entity_id",
                "entity_name",
                "Product",
                "Country",
                "Zone",
                "role",
                "import_share",
                "import_kbd_avg",
                "export_kbd_avg",
                "total_kb",
                "n_installations",
                "is_desk_complex",
            ]
            if c in roles_view.columns
        ]
    ],
    on=["entity_id", "entity_name", "Product"],
    how="left",
)

print(
    f"{FOCUS_COUNTRY} pattern rows ({AGGREGATE_BY}): {len(patterns):,} | "
    f"enough history: {int(patterns['enough_history'].sum()):,}"
)
print(
    "soft flags — import:",
    int(patterns["import_soft"].sum()),
    "| export:",
    int(patterns["export_soft"].sum()),
)
print("as-of month:", patterns["asof_month"].max())
print("\nimport YoY corr (enough history):")
print(patterns.loc[patterns["enough_history"], "import_yoy_corr"].describe())

soft_import = patterns.loc[
    patterns["import_soft"] & patterns["enough_history"]
].sort_values("import_z")
soft_export = patterns.loc[
    patterns["export_soft"] & patterns["enough_history"]
].sort_values("export_z")

print(f"\nSoft IMPORT (demand proxy) — last {RECENT_MONTHS}m:")
display(
    soft_import[
        [
            c
            for c in [
                "entity_name",
                "Product",
                "role",
                "import_recent_kbd",
                "import_hist_mean_kbd",
                "import_z",
                "import_seasonal_ratio",
                "import_yoy_corr",
                "role_stability",
                "n_installations",
            ]
            if c in soft_import.columns
        ]
    ].head(25)
)

print(f"\nSoft EXPORT (supply proxy) — last {RECENT_MONTHS}m:")
display(
    soft_export[
        [
            c
            for c in [
                "entity_name",
                "Product",
                "role",
                "export_recent_kbd",
                "export_hist_mean_kbd",
                "export_z",
                "export_seasonal_ratio",
                "export_yoy_corr",
                "role_stability",
                "n_installations",
            ]
            if c in soft_export.columns
        ]
    ].head(25)
)

example_names = EXAMPLE_COMPLEXES if AGGREGATE_BY == "complex" else EXAMPLE_INSTALLATIONS
print("\nExample entities — pattern snapshot:")
display(
    patterns.loc[patterns["entity_name"].isin(example_names)]
    .sort_values(["entity_name", "total_kb"], ascending=[True, False])[
        [
            c
            for c in [
                "entity_name",
                "Product",
                "role",
                "import_yoy_corr",
                "import_cv",
                "import_z",
                "import_seasonal_ratio",
                "import_soft",
                "export_z",
                "export_seasonal_ratio",
                "export_soft",
                "role_stability",
            ]
            if c in patterns.columns
        ]
    ]
)


Germany pattern rows (complex): 229 | enough history: 148
soft flags — import: 6 | export: 1
as-of month: 2026-08-01 00:00:00

import YoY corr (enough history):
count    105.000000
mean      -0.178088
std        0.203035
min       -1.000000
25%       -0.229170
50%       -0.153438
75%       -0.090947
max        0.393651
Name: import_yoy_corr, dtype: float64

Soft IMPORT (demand proxy) — last 3m:


,entity_name,Product,role,import_recent_kbd,import_hist_mean_kbd,import_z,import_seasonal_ratio,import_yoy_corr,role_stability,n_installations
15,BMÔ Bremen,Diesel,demand_center,3.650878,15.370879,-1.424155,0.217947,0.136925,0.833333,1
2,GOR Storage,Diesel,demand_center,3.154597,12.594871,-1.286128,0.247587,-0.253753,0.833333,2
50,Evos Hamburg,Diesel,demand_center,11.246354,25.695081,-1.231500,0.382763,0.023125,0.916667,1
145,Oiltank. Hamburg,Gasoil,demand_center,1.506200,11.133002,-1.082331,0.150930,-0.044570,0.416667,1
142,Oiltank. Hamburg,Diesel,demand_center,24.997052,39.097983,-0.840136,0.627657,-0.049450,1.000000,1
119,Nynas Hamburg Refinery,Diesel,demand_center,0.545600,5.321353,-0.552940,0.118420,-0.047276,0.750000,1



Soft EXPORT (supply proxy) — last 3m:


,entity_name,Product,role,export_recent_kbd,export_hist_mean_kbd,export_z,export_seasonal_ratio,export_yoy_corr,role_stability,n_installations
222,Wilhelmshaven Refinery,FO,supply_center,7.25452,17.737218,-0.984593,0.403189,-0.247349,1.0,2



Example entities — pattern snapshot:


,entity_name,Product,role,import_yoy_corr,import_cv,import_z,import_seasonal_ratio,import_soft,export_z,export_seasonal_ratio,export_soft,role_stability


### Pattern charts

1. **Predictability scatter** — YoY corr vs CV (lumpy vs seasonal)  
2. **Soft map** — recent import z vs seasonal ratio (demand softness)  
3. **Example site timelines** — monthly import/export kbd for `EXAMPLE_INSTALLATIONS`

In [23]:
pat_ok = patterns.loc[patterns["enough_history"]].copy()
pat_ok["label"] = pat_ok["entity_name"].astype(str) + " · " + pat_ok["Product"].astype(str)

# 1) Lumpy vs seasonal
fig_pred = px.scatter(
    pat_ok,
    x="import_cv",
    y="import_yoy_corr",
    color="role",
    size="total_kb",
    size_max=32,
    hover_name="label",
    color_discrete_map=ROLE_COLORS,
    title=f"{FOCUS_COUNTRY}: import predictability (YoY corr vs CV)",
    labels={
        "import_cv": "import CV (higher = lumpier)",
        "import_yoy_corr": "import YoY month corr (higher = more seasonal)",
    },
)
fig_pred.add_hline(y=0, line_dash="dot", line_color="#999")
fig_pred.update_layout(height=480)
fig_pred.show()

# 2) Soft demand monitor
soft_plot = pat_ok.loc[pat_ok["import_hist_mean_kbd"] >= MIN_SOFT_MEAN_KBD].copy()
soft_plot["soft_flag"] = soft_plot["import_soft"].map({True: "soft", False: "normal"})
fig_soft = px.scatter(
    soft_plot,
    x="import_seasonal_ratio",
    y="import_z",
    color="soft_flag",
    color_discrete_map={"soft": "#c45c26", "normal": "#9ca3af"},
    hover_name="label",
    hover_data={
        "role": True,
        "import_recent_kbd": ":.2f",
        "import_hist_mean_kbd": ":.2f",
        "import_seasonal_ratio": ":.2f",
        "import_z": ":.2f",
    },
    title=(
        f"{FOCUS_COUNTRY}: last-{RECENT_MONTHS}m import softness "
        f"(z≤{SOFT_Z} or seasonal_ratio≤{SOFT_SEASONAL_RATIO})"
    ),
    labels={
        "import_seasonal_ratio": "recent / same-calendar-month history",
        "import_z": "recent z vs all prior months",
    },
)
fig_soft.add_hline(y=SOFT_Z, line_dash="dash", line_color="#c45c26")
fig_soft.add_vline(x=SOFT_SEASONAL_RATIO, line_dash="dash", line_color="#c45c26")
fig_soft.update_layout(height=480)
fig_soft.show()

# 3) Example timelines (all products at discovery sites)
example_names = EXAMPLE_COMPLEXES if AGGREGATE_BY == "complex" else EXAMPLE_INSTALLATIONS
ex_m = monthly_focus.loc[monthly_focus["entity_name"].isin(example_names)].copy()
if ex_m.empty:
    print("No example entity rows in monthly_focus.")
else:
    ex_wide = (
        ex_m.groupby(
            ["entity_name", "Product", "year_month", "Imp/Exp"], dropna=False
        )["kbd"]
        .sum()
        .reset_index()
    )
    fig_ex = px.line(
        ex_wide,
        x="year_month",
        y="kbd",
        color="Product",
        line_dash="Imp/Exp",
        facet_row="entity_name",
        title=f"Example {AGGREGATE_BY} entities — monthly kbd by product (Imp/Exp)",
        height=280 * max(ex_wide["entity_name"].nunique(), 1),
    )
    fig_ex.update_yaxes(matches=None, title_text="kbd")
    fig_ex.update_layout(legend_title_text="")
    fig_ex.show()


No example entity rows in monthly_focus.


## 7) Owner OD flows — captive logistics & balancing (Sankey)

Waterborne **installation -> installation** flows from `kpler_trades.duckdb`
(the stacked Tableau master only keeps one `Installation` per row).

**Why this sits in notebook 04:** we join each OD edge to the lifetime
`Installation × Product` roles from §3, so a Repsol surplus site feeding a
Repsol deficit site shows up as internal balancing — not just a pretty Sankey.

**Inputs**
- Owners / primary_tag / tag flags / complexes: `data/manual/terminal_desk_map.csv` (desk map; edit freely)
- Filters: `OD_*` knobs in the setup cell (`OD_FOCUS_OWNER`, `OD_PRODUCTS`, …)

**Caveats**
- Ownership is **not** in Kpler — unmapped sites get `owner=(unmapped)`
- Pipe moves (e.g. Petronor ↔ Bilbao Oil Facilities) are invisible here
- Seller=Buyer is a captive proxy when the owner map is incomplete

Hub drill-down (`OD_HUB_ENTITY`): bipartite Sankey + tornado bars for one
installation/complex — who feeds the hub and who it ships to.

PyGWalker stays in notebook `03`; cells §9–§10 here remain commented.


In [24]:
# Desk owner map + raw OD edges (origin/destination both present)
owners = owners_view(load_desk_map(DESK_MAP_PATH))
owners["owner"] = owners["owner"].astype(str).str.strip().replace({"nan": "", "<NA>": ""})
owners["site_type"] = owners["site_type"].astype(str).str.strip().replace({"nan": "", "<NA>": ""})
owners.loc[owners["owner"].eq(""), "owner"] = pd.NA
owners.loc[owners["site_type"].eq(""), "site_type"] = pd.NA

assert DB_PATH.exists(), f"Missing {DB_PATH} — run bootstrap_duckdb.py first"

product_filter_sql = ""
if OD_PRODUCTS:
    in_list = ", ".join("'" + p.replace("'", "''") + "'" for p in OD_PRODUCTS)
    product_filter_sql = f"AND Product IN ({in_list})"


def _dest_country_clause(domestic_only: bool) -> str:
    # True keeps intra-country legs; False keeps cross-border only.
    if domestic_only:
        return 'AND "Country (destination)" = "Country (origin)"'
    return 'AND "Country (destination)" <> "Country (origin)"'


def _load_od_edges(*, direction: str, domestic_only: bool) -> pd.DataFrame:
    """Load installation→installation OD edges for the focus country.

    direction='export': origin country = focus country (outbound / leaving Spain)
    direction='import': destination country = focus country (inbound / into Spain)
    """
    if direction not in {"export", "import"}:
        raise ValueError(f"direction must be 'export' or 'import', got {direction!r}")

    country = OD_ORIGIN_COUNTRY.replace("'", "''")
    if direction == "export":
        country_filter = f"AND \"Country (origin)\" = '{country}'"
    else:
        country_filter = f"AND \"Country (destination)\" = '{country}'"

    dest_country_sql = _dest_country_clause(domestic_only)
    od_sql = f"""
SELECT
  "Installation origin" AS origin_installation,
  "Installation Destination" AS dest_installation,
  "Country (origin)" AS origin_country,
  "Country (destination)" AS dest_country,
  "Zone Origin" AS origin_zone,
  "Zone Destination" AS dest_zone,
  COALESCE(Product, '(unknown)') AS Product,
  COALESCE("Seller (origin)", '(unknown)') AS seller,
  COALESCE("Buyer (destination)", '(unknown)') AS buyer,
  COUNT(*) AS n_trades,
  SUM(TRY_CAST("Volume (bbl)" AS DOUBLE) / 1000.0) AS kb
FROM trades
WHERE "Trade status" = '{STATUS}'
  {country_filter}
  AND "Installation origin" IS NOT NULL AND TRIM("Installation origin") <> ''
  AND "Installation Destination" IS NOT NULL AND TRIM("Installation Destination") <> ''
  AND Family <> 'Dirty'
  AND Product NOT IN ('Crude', 'Condensate')
  AND "Group" <> 'Crude/Co'
  {dest_country_sql}
  {product_filter_sql}
GROUP BY 1, 2, 3, 4, 5, 6, 7, 8, 9
"""
    con_od = duckdb.connect(str(DB_PATH), read_only=True)
    try:
        return con_od.execute(od_sql).df()
    finally:
        con_od.close()


# Outbound from focus country (existing domestic + export Sankeys)
od_raw = _load_od_edges(direction="export", domestic_only=True)
print(
    f"raw outbound domestic OD edges: {len(od_raw):,} | kb={od_raw['kb'].sum():,.0f}"
)
print(f"products: {sorted(od_raw['Product'].dropna().unique().tolist())}")

od_raw_intl = _load_od_edges(direction="export", domestic_only=False)
print(
    f"raw outbound cross-border OD edges: {len(od_raw_intl):,} | "
    f"kb={od_raw_intl['kb'].sum():,.0f}"
)

# Inbound into focus country (foreign origin → Spanish destination)
od_raw_inbound_intl = _load_od_edges(direction="import", domestic_only=False)
print(
    f"raw inbound cross-border OD edges: {len(od_raw_inbound_intl):,} | "
    f"kb={od_raw_inbound_intl['kb'].sum():,.0f}"
)
od_raw.head(8)


raw outbound domestic OD edges: 60 | kb=4,529
products: ['FO', 'Gasoil', 'Gasoline', 'Jet', 'Naphtha']
raw outbound cross-border OD edges: 455 | kb=120,392
raw inbound cross-border OD edges: 661 | kb=206,810


,origin_installation,dest_installation,origin_country,dest_country,origin_zone,dest_zone,Product,seller,buyer,n_trades,kb
0,Mercuria Biofuels Brunsbuttel,Evos Hamburg,Germany,Germany,Brunsbuttel Port,Hamburg,Gasoil,Mercuria,(unknown),1,15.774
1,Yara Brunsbuttel,UTG Kiel,Germany,Germany,Brunsbuttel Port,Kiel,Jet,Yara,UTG,1,10.208
2,Omya Emden,Nynas Hamburg Refinery,Germany,Germany,Emden,Hamburg,FO,BMT SeaTech Ltd.,NYNAS AB,1,1.434
3,GÖR,GÖR,Germany,Germany,Rostock,Rostock,Gasoil,"Euroports,TotalEnergies","Euroports,TotalEnergies",1,3.302
4,HES Wilhelmshaven,HES Wilhelmshaven,Germany,Germany,Wilhelmshaven,Wilhelmshaven,Gasoil,(unknown),(unknown),1,8.912
5,Olenex Brake,Nynas Hamburg Refinery,Germany,Germany,Brake,Hamburg,FO,BMT SeaTech Ltd.,NYNAS AB,1,4.308
6,OAT Brunsbuttel,Oiltank. Hamburg,Germany,Germany,Brunsbuttel Port,Hamburg,Gasoil,OAT,(unknown),2,122.053
7,Holborn Hamburg Refinery,UTG Kiel,Germany,Germany,Hamburg,Kiel,FO,Oilinvest,UTG,1,11.944


In [25]:
def _attach_owner(df: pd.DataFrame, inst_col: str, prefix: str) -> pd.DataFrame:
    out = df.merge(
        owners.rename(
            columns={
                "installation": inst_col,
                "owner": f"{prefix}_owner",
                "site_type": f"{prefix}_site_type",
            }
        )[[inst_col, f"{prefix}_owner", f"{prefix}_site_type"]],
        on=inst_col,
        how="left",
    )
    out[f"{prefix}_owner"] = out[f"{prefix}_owner"].fillna("(unmapped)")
    out[f"{prefix}_site_type"] = out[f"{prefix}_site_type"].fillna("(unmapped)")
    return out


def classify_od_flow(row: pd.Series) -> str:
    # Prefer explicit same-owner tags; fall back to Seller=Buyer captive proxy.
    o_owner, d_owner = row["origin_owner"], row["dest_owner"]
    o_type, d_type = row["origin_site_type"], row["dest_site_type"]
    same_owner = o_owner != "(unmapped)" and o_owner == d_owner
    if same_owner and o_type == "refinery" and d_type == "refinery":
        return "internal_balancing"
    if same_owner:
        return "captive_same_owner"
    if row["seller"] not in {"(unknown)", ""} and row["seller"] == row["buyer"]:
        return "seller_buyer_match"
    if row["dest_zone"] in OD_ISLAND_ZONES:
        return "island_or_coastal_supply"
    return "commercial_to_3rd"


def build_od_flows(raw: pd.DataFrame) -> pd.DataFrame:
    flows = _attach_owner(raw, "origin_installation", "origin")
    flows = _attach_owner(flows, "dest_installation", "dest")
    flows["same_owner"] = (
        (flows["origin_owner"] != "(unmapped)")
        & (flows["origin_owner"] == flows["dest_owner"])
    )
    flows["seller_eq_buyer"] = (
        (flows["seller"] != "(unknown)") & (flows["seller"] == flows["buyer"])
    )
    flows["flow_class"] = flows.apply(classify_od_flow, axis=1)

    # Join §3 lifetime roles so we can flag surplus -> deficit product moves
    role_cols = ["Installation", "Product", "role", "import_share", "net_kb"]
    roles_od = roles[role_cols].copy()
    flows = flows.merge(
        roles_od.rename(
            columns={
                "Installation": "origin_installation",
                "role": "origin_role",
                "import_share": "origin_import_share",
                "net_kb": "origin_net_kb",
            }
        ),
        on=["origin_installation", "Product"],
        how="left",
    )
    flows = flows.merge(
        roles_od.rename(
            columns={
                "Installation": "dest_installation",
                "role": "dest_role",
                "import_share": "dest_import_share",
                "net_kb": "dest_net_kb",
            }
        ),
        on=["dest_installation", "Product"],
        how="left",
    )
    flows["surplus_to_deficit"] = (
        flows["origin_role"].eq("supply_center") & flows["dest_role"].eq("demand_center")
    )
    return flows


def filter_od_view(flows: pd.DataFrame) -> pd.DataFrame:
    view = flows.loc[flows["kb"] >= OD_MIN_EDGE_KB].copy()
    if OD_FOCUS_OWNER:
        view = view.loc[
            view["origin_owner"].eq(OD_FOCUS_OWNER)
            | view["dest_owner"].eq(OD_FOCUS_OWNER)
            | view["seller"].eq(OD_FOCUS_OWNER)
            | view["buyer"].eq(OD_FOCUS_OWNER)
        ].copy()
    return view


def summarize_od_view(view: pd.DataFrame, scope_label: str) -> None:
    view_kb = float(view["kb"].sum()) if len(view) else 0.0
    same_kb = (
        float(view.loc[view["same_owner"], "kb"].sum()) if len(view) else 0.0
    )
    s2d_kb = (
        float(view.loc[view["surplus_to_deficit"], "kb"].sum()) if len(view) else 0.0
    )

    print(
        f"OD view ({scope_label}) | owner={OD_FOCUS_OWNER!r} | "
        f"edges>={OD_MIN_EDGE_KB:g} kb: {len(view):,} | kb={view_kb:,.0f}"
    )
    print("\nflow_class mix:")
    print(
        view.groupby("flow_class", dropna=False)
        .agg(edges=("kb", "size"), kb=("kb", "sum"))
        .sort_values("kb", ascending=False)
    )
    print(
        f"\nsame_owner share of view kb: "
        f"{(100 * same_kb / view_kb) if view_kb else 0:.1f}%"
    )
    print(
        f"surplus->deficit share of view kb: "
        f"{(100 * s2d_kb / view_kb) if view_kb else 0:.1f}%"
    )


od_flows = build_od_flows(od_raw)
od_view = filter_od_view(od_flows)
summarize_od_view(od_view, "outbound domestic")

od_flows_intl = build_od_flows(od_raw_intl)
od_view_intl = filter_od_view(od_flows_intl)
print()
summarize_od_view(od_view_intl, "outbound cross-border")

od_flows_inbound_intl = build_od_flows(od_raw_inbound_intl)
od_view_inbound_intl = filter_od_view(od_flows_inbound_intl)
print()
summarize_od_view(od_view_inbound_intl, "inbound cross-border")

# Unified Spain-touched OD graph for hub drill-down (domestic + export + import legs).
od_raw_all = pd.concat(
    [od_raw, od_raw_intl, od_raw_inbound_intl], ignore_index=True
)
od_flows_all = build_od_flows(od_raw_all)


OD view (outbound domestic) | owner=None | edges>=500 kb: 3 | kb=2,178

flow_class mix:
                   edges        kb
flow_class                        
commercial_to_3rd      3  2177.889

same_owner share of view kb: 0.0%
surplus->deficit share of view kb: 76.2%

OD view (outbound cross-border) | owner=None | edges>=500 kb: 48 | kb=82,132

flow_class mix:
                   edges         kb
flow_class                         
commercial_to_3rd     48  82131.586

same_owner share of view kb: 0.0%
surplus->deficit share of view kb: 26.4%

OD view (inbound cross-border) | owner=None | edges>=500 kb: 86 | kb=137,929

flow_class mix:
                    edges          kb
flow_class                           
commercial_to_3rd      84  134493.586
seller_buyer_match      2    3435.413

same_owner share of view kb: 0.0%
surplus->deficit share of view kb: 86.0%


In [26]:
# Top OD edges + captive / balancing tables
show_cols = [
    "origin_installation",
    "dest_installation",
    "Product",
    "kb",
    "n_trades",
    "flow_class",
    "origin_owner",
    "dest_owner",
    "origin_role",
    "dest_role",
    "surplus_to_deficit",
    "seller",
    "buyer",
]

scope = f"{OD_ORIGIN_COUNTRY}{' domestic' if OD_DOMESTIC_ONLY else ''}"
top_od = od_view.sort_values("kb", ascending=False).head(30)
print(f"Top {len(top_od)} OD edges ({scope} | focus={OD_FOCUS_OWNER!r}):")
display(top_od[show_cols].reset_index(drop=True))

captive = od_view.loc[od_view["same_owner"] | od_view["seller_eq_buyer"]].sort_values(
    "kb", ascending=False
)
print(f"\nCaptive / same-owner / seller=buyer edges: {len(captive):,}")
display(captive[show_cols].head(25).reset_index(drop=True))

bal = od_view.loc[
    od_view["flow_class"].eq("internal_balancing") | od_view["surplus_to_deficit"]
].sort_values("kb", ascending=False)
print(f"\nInternal balancing / surplus->deficit edges: {len(bal):,}")
display(bal[show_cols].head(25).reset_index(drop=True))


Top 3 OD edges (Germany domestic | focus=None):


,origin_installation,dest_installation,Product,kb,n_trades,flow_class,origin_owner,dest_owner,origin_role,dest_role,surplus_to_deficit,seller,buyer
0,OAT Brunsbuttel,BMÔ Bremen,Gasoline,921.760,22,commercial_to_3rd,Heide refinery,BMO,supply_center,demand_center,True,OAT,BMÔ
1,HES Wilhelmshaven,Evos Hamburg,Gasoil,738.403,4,commercial_to_3rd,HES International,Evos,supply_center,demand_center,True,(unknown),(unknown)
2,Dow Stade,Oiltank. Hamburg,Naphtha,517.726,1,commercial_to_3rd,Dow,MB Energy,supply_center,balanced_hub,False,DOW,(unknown)



Captive / same-owner / seller=buyer edges: 0


,origin_installation,dest_installation,Product,kb,n_trades,flow_class,origin_owner,dest_owner,origin_role,dest_role,surplus_to_deficit,seller,buyer



Internal balancing / surplus->deficit edges: 2


,origin_installation,dest_installation,Product,kb,n_trades,flow_class,origin_owner,dest_owner,origin_role,dest_role,surplus_to_deficit,seller,buyer
0,OAT Brunsbuttel,BMÔ Bremen,Gasoline,921.760,22,commercial_to_3rd,Heide refinery,BMO,supply_center,demand_center,True,OAT,BMÔ
1,HES Wilhelmshaven,Evos Hamburg,Gasoil,738.403,4,commercial_to_3rd,HES International,Evos,supply_center,demand_center,True,(unknown),(unknown)


In [27]:
def plot_od_sankey(
    edges: pd.DataFrame,
    *,
    title: str,
    top_n: int = OD_SANKEY_TOP_N,
    source_col: str = "origin_installation",
    target_col: str = "dest_installation",
    value_col: str = "kb",
    label_col: str = "Product",
    show_country: bool = False,
) -> go.Figure:
    """Compact Sankey: top_n links, node labels = installation (+ owner if mapped)."""
    plot_df = edges.nlargest(top_n, value_col).copy()
    if plot_df.empty:
        fig = go.Figure()
        fig.update_layout(title=title + " (no edges)")
        return fig

    def _node_label(inst: str, owner: str, country: str | None = None) -> str:
        base = f"{inst} [{owner}]" if owner and owner != "(unmapped)" else str(inst)
        if show_country and country:
            return f"{base} ({country})"
        return base

    nodes = pd.Index(
        pd.concat([plot_df[source_col], plot_df[target_col]], ignore_index=True).unique()
    )
    node_map = {n: i for i, n in enumerate(nodes)}

    owner_lookup = (
        pd.concat(
            [
                plot_df[[source_col, "origin_owner"]].rename(
                    columns={source_col: "installation", "origin_owner": "owner"}
                ),
                plot_df[[target_col, "dest_owner"]].rename(
                    columns={target_col: "installation", "dest_owner": "owner"}
                ),
            ],
            ignore_index=True,
        )
        .drop_duplicates("installation")
        .set_index("installation")["owner"]
    )
    country_lookup = {}
    if show_country:
        country_lookup = (
            pd.concat(
                [
                    plot_df[[source_col, "origin_country"]].rename(
                        columns={
                            source_col: "installation",
                            "origin_country": "country",
                        }
                    ),
                    plot_df[[target_col, "dest_country"]].rename(
                        columns={
                            target_col: "installation",
                            "dest_country": "country",
                        }
                    ),
                ],
                ignore_index=True,
            )
            .drop_duplicates("installation")
            .set_index("installation")["country"]
            .to_dict()
        )
    labels = [
        _node_label(
            n,
            owner_lookup.get(n, "(unmapped)"),
            country_lookup.get(n),
        )
        for n in nodes
    ]

    link_colors = []
    for fc in plot_df["flow_class"]:
        c = FLOW_CLASS_COLORS.get(fc, "#9ca3af")
        if isinstance(c, str) and c.startswith("#") and len(c) == 7:
            link_colors.append(
                f"rgba({int(c[1:3], 16)},{int(c[3:5], 16)},{int(c[5:7], 16)},0.45)"
            )
        else:
            link_colors.append("rgba(156,163,175,0.45)")

    fig = go.Figure(
        data=[
            go.Sankey(
                arrangement="snap",
                node=dict(
                    pad=14,
                    thickness=14,
                    line=dict(color="#374151", width=0.5),
                    label=labels,
                    color="#e5e7eb",
                ),
                link=dict(
                    source=plot_df[source_col].map(node_map),
                    target=plot_df[target_col].map(node_map),
                    value=plot_df[value_col],
                    label=(
                        plot_df[label_col].astype(str)
                        + " · "
                        + plot_df["flow_class"].astype(str)
                        + " · "
                        + plot_df[value_col].round(0).astype(int).astype(str)
                        + " kb"
                    ),
                    color=link_colors,
                ),
            )
        ]
    )
    fig.update_layout(
        title=title,
        font=dict(size=11),
        height=650,
        margin=dict(l=20, r=20, t=60, b=20),
    )
    return fig


owner_label = OD_FOCUS_OWNER or "all owners"
fig_sankey = plot_od_sankey(
    od_view,
    title=(
        f"{OD_ORIGIN_COUNTRY} outbound domestic OD Sankey — {owner_label} "
        f"(top {OD_SANKEY_TOP_N} edges >= {OD_MIN_EDGE_KB:g} kb)"
    ),
)
fig_sankey.show()

# Product-split Sankeys for the focus owner (keeps each chart readable)
if OD_FOCUS_OWNER and OD_PRODUCTS:
    for prod in OD_PRODUCTS:
        sub = od_view.loc[od_view["Product"].eq(prod)]
        if sub.empty:
            continue
        fig_p = plot_od_sankey(
            sub,
            title=f"{OD_ORIGIN_COUNTRY} · {OD_FOCUS_OWNER} · {prod}",
            top_n=min(15, OD_SANKEY_TOP_N),
        )
        fig_p.show()

# Outbound cross-border: Spain origin → foreign destination (exports).
fig_sankey_intl = plot_od_sankey(
    od_view_intl,
    title=(
        f"{OD_ORIGIN_COUNTRY} outbound cross-border OD Sankey — {owner_label} "
        f"(top {OD_SANKEY_TOP_N} edges >= {OD_MIN_EDGE_KB:g} kb)"
    ),
)
fig_sankey_intl.show()

# Inbound cross-border: foreign origin → Spain destination (imports).
fig_sankey_inbound = plot_od_sankey(
    od_view_inbound_intl,
    title=(
        f"Into {OD_ORIGIN_COUNTRY} inbound cross-border OD Sankey — {owner_label} "
        f"(top {OD_SANKEY_TOP_N} edges >= {OD_MIN_EDGE_KB:g} kb)"
    ),
    show_country=True,
)
fig_sankey_inbound.show()


### Hub drill-down — inbound vs outbound for one site/complex

Set `OD_HUB_ENTITY` in the setup cell (installation name, `complex_id`, or desk
`complex_name`). The chart puts the **hub in the centre**: partners on the left
feed it (**inbound**), partners on the right receive from it (**outbound**).

Counterparties roll up to desk **complex** by default (`OD_HUB_COUNTERPARTY_GRAIN`)
so e.g. Bilbao refinery + port appear as one partner. Intra-complex legs are
excluded.

**OD** = **Origin–Destination** (installation→installation trade legs in Kpler).


In [ ]:
OD_HUB_ENTITY = "Barcelona 32C"  # installation, complex_id, or complex_name; None = skip
OD_HUB_GRAIN = "installation"  # resolve hub as "installation" or "complex"
OD_HUB_COUNTERPARTY_GRAIN = "complex"  # roll partners up to desk complex when mapped
OD_HUB_TOP_N = 12  # max counterparty×product links per direction in hub charts

In [28]:
def resolve_hub_entity(
    hub_entity: str,
    *,
    grain: str,
) -> tuple[str, str, set[str]]:
    """Return hub_id, display label, and member installations."""
    if grain == "complex":
        mask = entity_map["complex_id"].eq(hub_entity) | entity_map["complex_name"].eq(
            hub_entity
        )
        members_df = entity_map.loc[mask]
        if members_df.empty:
            raise ValueError(f"Unknown complex hub: {hub_entity!r}")
        hub_id = members_df["complex_id"].iloc[0]
        hub_label = members_df["complex_name"].iloc[0]
        members = set(members_df["Installation"].astype(str))
        return hub_id, hub_label, members

    hub_entity = str(hub_entity).strip()
    members_df = entity_map.loc[entity_map["Installation"].eq(hub_entity)]
    if members_df.empty:
        return f"inst:{hub_entity}", hub_entity, {hub_entity}
    row = members_df.iloc[0]
    return row["complex_id"], row["Installation"], {row["Installation"]}


def _counterparty_label(installation: str) -> str:
    inst = str(installation).strip()
    if OD_HUB_COUNTERPARTY_GRAIN == "complex":
        hit = entity_map.loc[entity_map["Installation"].eq(inst)]
        if len(hit):
            return str(hit["complex_name"].iloc[0])
    return inst


def build_hub_trade_edges(
    flows: pd.DataFrame,
    hub_members: set[str],
    *,
    direction: str,
    top_n: int = OD_HUB_TOP_N,
) -> pd.DataFrame:
    """Aggregate external inbound/outbound legs for a hub (counterparty × product)."""
    hub_members = {str(x) for x in hub_members}
    sub = flows.loc[flows["kb"] >= OD_MIN_EDGE_KB].copy()
    if direction == "inbound":
        sub = sub.loc[sub["dest_installation"].astype(str).isin(hub_members)]
        sub = sub.loc[~sub["origin_installation"].astype(str).isin(hub_members)]
        cp_col = "origin_installation"
    elif direction == "outbound":
        sub = sub.loc[sub["origin_installation"].astype(str).isin(hub_members)]
        sub = sub.loc[~sub["dest_installation"].astype(str).isin(hub_members)]
        cp_col = "dest_installation"
    else:
        raise ValueError(f"direction must be 'inbound' or 'outbound', got {direction!r}")

    if sub.empty:
        return pd.DataFrame(
            columns=["counterparty", "Product", "kb", "n_trades"]
        )

    sub["counterparty"] = sub[cp_col].astype(str).map(_counterparty_label)
    agg = (
        sub.groupby(["counterparty", "Product"], as_index=False)
        .agg(kb=("kb", "sum"), n_trades=("n_trades", "sum"))
        .sort_values("kb", ascending=False)
    )
    if OD_PRODUCTS:
        agg = agg.loc[agg["Product"].isin(OD_PRODUCTS)]
    return agg.head(top_n).reset_index(drop=True)


def _product_rgba(product: str, alpha: float = 0.55) -> str:
    c = PRODUCT_COLORS.get(product, "#9ca3af")
    return f"rgba({int(c[1:3], 16)},{int(c[3:5], 16)},{int(c[5:7], 16)},{alpha})"


def plot_hub_trade_sankey(
    hub_label: str,
    inbound: pd.DataFrame,
    outbound: pd.DataFrame,
    *,
    title: str,
) -> go.Figure:
    """Bipartite Sankey: partners → hub → partners."""
    if inbound.empty and outbound.empty:
        fig = go.Figure()
        fig.update_layout(title=title + " (no edges)")
        return fig

    left_nodes = inbound["counterparty"].drop_duplicates().tolist() if len(inbound) else []
    right_nodes = (
        outbound["counterparty"].drop_duplicates().tolist() if len(outbound) else []
    )
    nodes = left_nodes + [hub_label] + right_nodes
    hub_idx = len(left_nodes)
    node_map = {n: i for i, n in enumerate(nodes)}

    sources, targets, values, link_labels, link_colors = [], [], [], [], []

    for _, row in inbound.iterrows():
        sources.append(node_map[row["counterparty"]])
        targets.append(hub_idx)
        values.append(row["kb"])
        link_labels.append(
            f"{row['Product']} · {row['kb']:,.0f} kb · {int(row['n_trades'])} trades"
        )
        link_colors.append(_product_rgba(row["Product"]))

    for _, row in outbound.iterrows():
        sources.append(hub_idx)
        targets.append(node_map[row["counterparty"]])
        values.append(row["kb"])
        link_labels.append(
            f"{row['Product']} · {row['kb']:,.0f} kb · {int(row['n_trades'])} trades"
        )
        link_colors.append(_product_rgba(row["Product"]))

    fig = go.Figure(
        data=[
            go.Sankey(
                arrangement="snap",
                node=dict(
                    pad=16,
                    thickness=16,
                    line=dict(color="#374151", width=0.5),
                    label=nodes,
                    color=["#dbeafe"] * len(left_nodes)
                    + ["#fcd34d"]
                    + ["#dcfce7"] * len(right_nodes),
                ),
                link=dict(
                    source=sources,
                    target=targets,
                    value=values,
                    label=link_labels,
                    color=link_colors,
                ),
            )
        ]
    )
    fig.update_layout(
        title=title,
        font=dict(size=11),
        height=700,
        margin=dict(l=20, r=20, t=60, b=20),
    )
    return fig


def plot_hub_trade_bars(
    inbound: pd.DataFrame,
    outbound: pd.DataFrame,
    *,
    title: str,
    top_partners: int = 6,
) -> go.Figure:
    """Tornado bars: inbound kb left (negative), outbound kb right (positive)."""
    rows: list[dict] = []
    products = sorted(
        set(inbound.get("Product", pd.Series(dtype=str)).tolist())
        | set(outbound.get("Product", pd.Series(dtype=str)).tolist())
    )

    for prod in products:
        for direction, df, sign in (
            ("inbound", inbound, -1),
            ("outbound", outbound, 1),
        ):
            sub = df.loc[df["Product"].eq(prod)] if len(df) else df
            if sub.empty:
                continue
            top = (
                sub.groupby("counterparty", as_index=False)["kb"]
                .sum()
                .nlargest(top_partners, "kb")
            )
            for _, row in top.iterrows():
                arrow = "←" if direction == "inbound" else "→"
                rows.append(
                    {
                        "y_label": f"{prod} {arrow} {row['counterparty']}",
                        "x": sign * row["kb"],
                        "kb": row["kb"],
                        "Product": prod,
                        "direction": direction,
                    }
                )

    if not rows:
        fig = go.Figure()
        fig.update_layout(title=title + " (no edges)")
        return fig

    plot_df = pd.DataFrame(rows).sort_values("x")
    colors = [_product_rgba(p, alpha=0.8) for p in plot_df["Product"]]

    fig = go.Figure(
        go.Bar(
            y=plot_df["y_label"],
            x=plot_df["x"],
            orientation="h",
            marker_color=colors,
            text=plot_df["kb"].map(lambda v: f"{v:,.0f} kb"),
            textposition="outside",
            hovertemplate="%{y}<br>%{text}<extra></extra>",
        )
    )
    fig.update_layout(
        title=title,
        xaxis_title="kb (inbound ← | → outbound)",
        yaxis_title="",
        height=max(420, 28 * len(plot_df)),
        margin=dict(l=20, r=20, t=60, b=20),
    )
    fig.add_vline(x=0, line_width=1, line_color="#374151")
    return fig


if OD_HUB_ENTITY:
    hub_id, hub_label, hub_members = resolve_hub_entity(
        OD_HUB_ENTITY, grain=OD_HUB_GRAIN
    )
    hub_inbound = build_hub_trade_edges(
        od_flows_all, hub_members, direction="inbound"
    )
    hub_outbound = build_hub_trade_edges(
        od_flows_all, hub_members, direction="outbound"
    )

    print(
        f"Hub drill-down | {hub_label} ({OD_HUB_GRAIN}; "
        f"{len(hub_members)} member site(s)) | "
        f"inbound links={len(hub_inbound)} | outbound links={len(hub_outbound)}"
    )
    display(
        pd.concat(
            [
                hub_inbound.assign(direction="inbound"),
                hub_outbound.assign(direction="outbound"),
            ],
            ignore_index=True,
        ).sort_values("kb", ascending=False)
    )

    fig_hub = plot_hub_trade_sankey(
        hub_label,
        hub_inbound,
        hub_outbound,
        title=(
            f"Hub trade Sankey — {hub_label} "
            f"(top {OD_HUB_TOP_N}/dir; partners at {OD_HUB_COUNTERPARTY_GRAIN} grain)"
        ),
    )
    fig_hub.show()

    fig_hub_bars = plot_hub_trade_bars(
        hub_inbound,
        hub_outbound,
        title=f"Hub trade profile — {hub_label} (inbound vs outbound by product)",
    )
    fig_hub_bars.show()
else:
    print("OD_HUB_ENTITY is None — skip hub drill-down charts.")


Hub drill-down | Repsol Coruna Refinery (installation; 1 member site(s)) | inbound links=0 | outbound links=1


C:\Users\luiscarlos.gaitan\AppData\Local\Temp\ipykernel_47680\963285427.py:237: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pd.concat(


,counterparty,Product,kb,n_trades,direction
0,GOR Storage,Naphtha,761.599,4,outbound


In [29]:
# Destination import mix: how much of a sink's inbound kb is same-owner / captive?
dest_mix = (
    od_flows.groupby(
        ["dest_installation", "dest_owner", "Product", "origin_owner"],
        dropna=False,
    )
    .agg(kb=("kb", "sum"), n_trades=("n_trades", "sum"))
    .reset_index()
)
dest_tot = dest_mix.groupby(["dest_installation", "Product"], as_index=False)["kb"].sum()
dest_tot = dest_tot.rename(columns={"kb": "dest_product_kb"})
dest_mix = dest_mix.merge(dest_tot, on=["dest_installation", "Product"], how="left")
dest_mix["origin_share"] = dest_mix["kb"] / dest_mix["dest_product_kb"].replace(0, pd.NA)

# Focus: destinations touched by OD_FOCUS_OWNER as origin, or owned by them
if OD_FOCUS_OWNER:
    focus_dests = set(
        od_flows.loc[
            od_flows["origin_owner"].eq(OD_FOCUS_OWNER)
            | od_flows["dest_owner"].eq(OD_FOCUS_OWNER),
            "dest_installation",
        ]
    )
    mix_view = dest_mix.loc[
        dest_mix["dest_installation"].isin(focus_dests)
        & (dest_mix["dest_product_kb"] >= OD_MIN_EDGE_KB)
    ].copy()
else:
    mix_view = dest_mix.loc[dest_mix["dest_product_kb"] >= OD_MIN_EDGE_KB].copy()

mix_view = mix_view.sort_values(
    ["dest_installation", "Product", "kb"], ascending=[True, True, False]
)
print("Destination × product import mix by origin_owner (share of inbound kb):")
display(
    mix_view.head(60)[
        [
            "dest_installation",
            "dest_owner",
            "Product",
            "origin_owner",
            "kb",
            "origin_share",
            "dest_product_kb",
        ]
    ].reset_index(drop=True)
)

# Heatmap: origin zone -> dest zone for focus owner
owner_label = OD_FOCUS_OWNER or "all owners"
heat = od_view.copy()
if len(heat):
    heat_agg = (
        heat.groupby(["origin_zone", "dest_zone"], as_index=False)["kb"]
        .sum()
        .sort_values("kb", ascending=False)
    )
    pivot = heat_agg.pivot(index="origin_zone", columns="dest_zone", values="kb").fillna(0)
    fig_heat = px.imshow(
        pivot,
        labels=dict(x="Destination zone", y="Origin zone", color="kb"),
        title=f"{OD_ORIGIN_COUNTRY} zone->zone kb ({owner_label})",
        aspect="auto",
        color_continuous_scale="YlOrRd",
    )
    fig_heat.update_layout(height=520, margin=dict(l=40, r=20, t=60, b=40))
    fig_heat.show()
else:
    print("No edges in od_view for zone heatmap.")

print(
    "\nEdit DESK_MAP_PATH / OD_FOCUS_OWNER / OD_PRODUCTS in setup, re-run this section, "
    "then re-run Persist to refresh owner_od_flows.*"
)


Destination × product import mix by origin_owner (share of inbound kb):


,dest_installation,dest_owner,Product,origin_owner,kb,origin_share,dest_product_kb
0,BMÔ Bremen,BMO,Gasoline,Heide refinery,921.760,0.850220,1084.143
1,BMÔ Bremen,BMO,Gasoline,Oilinvest Group,162.383,0.149780,1084.143
2,Evos Hamburg,Evos,Gasoil,HES International,738.403,0.967386,763.297
3,Evos Hamburg,Evos,Gasoil,Mercuria,15.774,0.020666,763.297
4,Evos Hamburg,Evos,Gasoil,Omya,9.120,0.011948,763.297
5,Oiltank. Hamburg,MB Energy,Naphtha,Dow,517.726,0.980607,527.965
6,Oiltank. Hamburg,MB Energy,Naphtha,MB Energy,10.239,0.019393,527.965



Edit DESK_MAP_PATH / OD_FOCUS_OWNER / OD_PRODUCTS in setup, re-run this section, then re-run Persist to refresh owner_od_flows.*


## 8) Persist derived tables

Re-run after changing thresholds, complex map, pattern knobs, OD knobs, or the master extract.


In [30]:
DERIVED_DIR.mkdir(parents=True, exist_ok=True)

monthly.to_parquet(OUT_MONTHLY, index=False)
roles.to_parquet(OUT_ROLES, index=False)
roles.to_csv(OUT_ROLES_CSV, index=False)
complex_roles.to_parquet(OUT_COMPLEX_ROLES, index=False)
complex_roles.to_csv(OUT_COMPLEX_ROLES_CSV, index=False)
patterns.to_parquet(OUT_PATTERNS, index=False)
patterns.to_csv(OUT_PATTERNS_CSV, index=False)
entity_map.to_csv(OUT_ENTITY_MAP, index=False)
if "od_flows" in globals() and len(od_flows):
    od_flows.to_parquet(OUT_OD_FLOWS, index=False)
    od_flows.to_csv(OUT_OD_FLOWS_CSV, index=False)

print("wrote", OUT_MONTHLY.name, f"({len(monthly):,} rows)")
print("wrote", OUT_ROLES.name, f"({len(roles):,} rows)")
print("wrote", OUT_ROLES_CSV.name)
print("wrote", OUT_COMPLEX_ROLES.name, f"({len(complex_roles):,} rows)")
print("wrote", OUT_COMPLEX_ROLES_CSV.name)
print("wrote", OUT_PATTERNS.name, f"({len(patterns):,} rows)")
print("wrote", OUT_PATTERNS_CSV.name)
print("wrote", OUT_ENTITY_MAP.name, f"({len(entity_map):,} rows)")
if "od_flows" in globals() and len(od_flows):
    print("wrote", OUT_OD_FLOWS.name, f"({len(od_flows):,} rows)")
    print("wrote", OUT_OD_FLOWS_CSV.name)
else:
    print("skipped OD flows (run §7 first, or empty result)")
print("AGGREGATE_BY used for patterns/roles_view:", AGGREGATE_BY)


wrote terminal_product_monthly.parquet (451,229 rows)
wrote terminal_product_roles.parquet (17,807 rows)
wrote terminal_product_roles.csv
wrote complex_product_roles.parquet (17,587 rows)
wrote complex_product_roles.csv
wrote terminal_product_patterns.parquet (229 rows)
wrote terminal_product_patterns.csv
wrote terminal_entity_map.csv (3,084 rows)
wrote owner_od_flows.parquet (60 rows)
wrote owner_od_flows.csv
AGGREGATE_BY used for patterns/roles_view: complex


## 9) PyGWalker (optional — prefer notebook 03) — monthly explorer (recommended)

Same grain as your discovery chart, for **any** product / site:

1. Filter `Country` = Spain (or pick installations / a `complex_name`)
2. X = `year_month`
3. Y = `Imp/Exp` + `sum(kbd_signed)` (or `sum(kb_signed)`)
4. Color = `Product`  
   Optional: also filter `Product` once you care about one grade


In [31]:
#walker_monthly = pyg.walk(
#    monthly,
#    kernel_computation=True,
#    default_tab="vis",
#)
#walker_monthly

## 10) PyGWalker (optional — prefer notebook 03) — lifetime roles (`roles_view`)

Uses the selected grain (`AGGREGATE_BY`):

1. Filter `Country` = Spain
2. Filter `role` ∈ demand_center / supply_center
3. X = `entity_name`, Y = `import_kbd_avg` / `export_kbd_avg`, Color = `Product`


In [32]:
#walker_roles = pyg.walk(
#    roles_view,
#    kernel_computation=True,
#    default_tab="vis",
#)
#walker_roles
